In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd

In [2]:

# --- PATH CONFIGURATION ---
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
# Modified to the requested folder name
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 2500
DT  = 1 / FPS

In [3]:
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)  # n not np
    n[n == 0] = 1
    return v / n

def turning_angle_3d(v):
    v1 = v[:-1]   # v(t-1)
    v2 = v[1:]    # v(t)

    dot = []
    cross = []

    for a, b in zip(v1, v2):
        d = np.dot(a, b)
        c = np.cross(a, b)
        dot.append(d)
        cross.append(c)

    dot = np.array(dot)
    cross = np.linalg.norm(cross, axis=1)

    angle = np.degrees(np.arctan2(cross, dot))
    return angle


def turning_angle_2d(v):
    v1 = v[:-1]
    v2 = v[1:]

    dot = []
    cross = []

    for a, b in zip(v1, v2):
        d = np.dot(a, b)
        c = np.cross(a, b)
        dot.append(d)
        cross.append(c)

    dot = np.array(dot)
    cross = np.array(cross)

    angle = np.degrees(np.arctan2(cross, dot))
    return angle

In [4]:

csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
for path in csv_files:
    fname = os.path.basename(path)
    print(f"Processing: {fname}")
    df = pd.read_csv(path)

    center = df[["center_X","center_Y","center_Z"]].values

    # Displacement between consecutive center positions
    displacement = np.diff(center, axis=0)       
    displacement = normalize(displacement)       

    # Turning angles (already effectively centered)
    turn_3d = turning_angle_3d(displacement)                            
    turn_xy = turning_angle_2d(normalize(displacement[:, :2]))          
    turn_yz = turning_angle_2d(normalize(displacement[:, 1:]))

    # FRAME ALIGNMENT
    frames = df["frame"].values[1:-1]

    out_df = pd.DataFrame({
        "frame": frames,
        "time_sec": frames / FPS,
        "turning_angle_deg": turn_3d,
        "turning_angle_xy_deg": turn_xy,
        "turning_angle_yz_deg": turn_yz,
        "turning_velocity_deg_s": turn_3d / DT,
    })

    out_name = fname.replace("_SMOOTH.csv", "_CENTER_TURNING_ANGLE.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print(f"Saved: {out_name}")

Processing: Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Trial2_180rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Trial2_200rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Trial3_180rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Trial4_400rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Trial5_180rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Trial5_400rpmxyzpts_CENTER_TURNING_ANGLE.csv
Processing: Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Trial7_400rpmxyzpts_CENTER_TURNING_ANGLE.csv
